# mini GPT Report Pipeline

이 노트북은 리포트용 실험을 위해 BPE 학습, GPT 학습, checkpoint 로드 테스트를 분리해서 실행합니다.

흐름:

1. NSMC 데이터 준비
2. BPE tokenizer 학습 및 저장
3. 저장된 token id를 로드해서 GPT 학습
4. epoch별 loss 표와 그래프 저장
5. best checkpoint를 다시 로드해서 validation/test/generation 테스트


## 0. Runtime Check

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("GPU runtime을 쓰는 것이 좋습니다: Runtime > Change runtime type > GPU")


## 1. Project Setup

In [ ]:
from pathlib import Path
import os
import subprocess

# Colab에 업로드한 프로젝트 폴더 경로입니다. 필요하면 바꾸세요.
PROJECT_DIR = "/content/gpt-lab"

# 리포트용 산출물 저장 위치입니다. Drive를 쓰면 런타임이 끊겨도 결과가 남습니다.
OUT = "/content/drive/MyDrive/gpt-lab-report-v3000"
SMOKE_OUT = "outputs/colab_pipeline_smoke"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT:", OUT)


In [ ]:
os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

!pip install -q -r requirements.txt


## 2. Unit Tests

In [ ]:
!python -m pytest tests/test_bpe.py tests/test_dataset.py tests/test_model.py tests/test_train.py -q


## 3. Smoke Test: BPE 학습

작은 데이터로 전체 파이프라인이 깨지지 않는지 먼저 확인합니다.

In [ ]:
subprocess.run([
    "python", "scripts/step1_train_bpe.py",
    "--out-dir", SMOKE_OUT,
    "--vocab-size", "300",
    "--train-chars", "20000",
    "--val-chars", "5000",
], check=True)


In [ ]:
import json
from IPython.display import display

manifest_path = Path(SMOKE_OUT) / "bpe_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
display(manifest)


## 4. Smoke Test: GPT 학습

In [ ]:
subprocess.run([
    "python", "scripts/step2_train_gpt.py",
    "--artifact-dir", SMOKE_OUT,
    "--run-name", "smoke",
    "--num-epochs", "1",
    "--context-length", "64",
    "--emb-dim", "64",
    "--n-heads", "4",
    "--n-layers", "2",
    "--batch-size", "8",
    "--eval-freq", "20",
    "--eval-iter", "2",
    "--train-token-limit", "20000",
    "--val-token-limit", "5000",
    "--scheduler", "cosine",
], check=True)


## 5. Smoke Test: 저장된 모델 로드 후 테스트

In [ ]:
subprocess.run([
    "python", "scripts/step3_test_gpt.py",
    "--artifact-dir", SMOKE_OUT,
    "--run-name", "smoke",
    "--eval-batches", "5",
    "--prompt", "이 영화",
], check=True)


In [ ]:
import pandas as pd
from IPython.display import Image

display(pd.read_csv(Path(SMOKE_OUT) / "runs/smoke/epoch_summary.csv"))
display(pd.read_csv(Path(SMOKE_OUT) / "runs/smoke/test/split_metrics.csv"))
display(Image(str(Path(SMOKE_OUT) / "runs/smoke/loss_curves.png")))


## 6. Drive Mount

아래부터는 리포트용 장기 실험입니다. Drive에 저장해야 런타임이 끊겨도 산출물이 남습니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Path(OUT).mkdir(parents=True, exist_ok=True)
print("OUT:", OUT)


## 7. Full BPE: vocab_size=3000

이 단계는 BPE tokenizer와 train/val token ids를 저장합니다. 이후 학습 단계는 이 산출물을 로드해서 사용합니다.

In [ ]:
subprocess.run([
    "python", "scripts/step1_train_bpe.py",
    "--out-dir", OUT,
    "--vocab-size", "3000",
], check=True)


In [ ]:
manifest = json.loads((Path(OUT) / "bpe_manifest.json").read_text(encoding="utf-8"))
print("actual_vocab_size:", manifest["actual_vocab_size"])
print("train_tokens:", manifest["train_tokens"])
print("val_tokens:", manifest["val_tokens"])
print("vocab_path:", manifest["vocab_path"])


## 8. Full Train: 100 Epoch Baseline

기준 실험입니다. scheduler 없이 기존 설정과 비슷하게 돌립니다.

In [ ]:
BASELINE_RUN = "baseline_v3000_e100_lr3e4_do01"

subprocess.run([
    "python", "scripts/step2_train_gpt.py",
    "--artifact-dir", OUT,
    "--run-name", BASELINE_RUN,
    "--num-epochs", "100",
    "--context-length", "128",
    "--emb-dim", "192",
    "--n-heads", "4",
    "--n-layers", "4",
    "--batch-size", "16",
    "--lr", "3e-4",
    "--drop-rate", "0.1",
    "--weight-decay", "0.1",
    "--scheduler", "none",
    "--eval-freq", "100",
    "--eval-iter", "20",
    "--grad-clip", "1.0",
], check=True)


## 9. Full Train: Improved Branch

과적합 완화를 확인하기 위한 비교 실험입니다. dropout을 높이고 cosine scheduler를 적용합니다.

In [ ]:
IMPROVED_RUN = "improved_v3000_e100_lr3e4_do02_cosine"

subprocess.run([
    "python", "scripts/step2_train_gpt.py",
    "--artifact-dir", OUT,
    "--run-name", IMPROVED_RUN,
    "--num-epochs", "100",
    "--context-length", "128",
    "--emb-dim", "192",
    "--n-heads", "4",
    "--n-layers", "4",
    "--batch-size", "16",
    "--lr", "3e-4",
    "--min-lr", "1e-5",
    "--drop-rate", "0.2",
    "--weight-decay", "0.1",
    "--scheduler", "cosine",
    "--eval-freq", "100",
    "--eval-iter", "20",
    "--grad-clip", "1.0",
], check=True)


## 10. Hyperparameter Branch: Learning Rate

리포트에 분기점으로 넣기 좋은 30 epoch 비교 실험입니다.

In [ ]:
LRS = ["1e-4", "3e-4", "5e-4"]

for lr in LRS:
    run_name = f"branch_lr_{lr}"
    print("running", run_name)
    subprocess.run([
        "python", "scripts/step2_train_gpt.py",
        "--artifact-dir", OUT,
        "--run-name", run_name,
        "--num-epochs", "30",
        "--context-length", "128",
        "--emb-dim", "192",
        "--n-heads", "4",
        "--n-layers", "4",
        "--batch-size", "16",
        "--lr", lr,
        "--drop-rate", "0.1",
        "--weight-decay", "0.1",
        "--scheduler", "cosine",
        "--eval-freq", "100",
        "--eval-iter", "20",
        "--grad-clip", "1.0",
    ], check=True)


## 11. 결과 표와 그래프 확인

In [ ]:
from IPython.display import Image, display

RUN = IMPROVED_RUN
run_dir = Path(OUT) / "runs" / RUN

epoch_df = pd.read_csv(run_dir / "epoch_summary.csv")
display(epoch_df.tail(20))
display(Image(str(run_dir / "loss_curves.png")))
display(Image(str(run_dir / "lr_by_epoch.png")))


In [ ]:
def summarize_run(run_name):
    path = Path(OUT) / "runs" / run_name / "epoch_summary.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    best_idx = df["val_loss"].idxmin()
    best = df.loc[best_idx]
    final = df.iloc[-1]
    return {
        "run": run_name,
        "epochs": int(final["epoch"]),
        "best_epoch": int(best["epoch"]),
        "best_val_loss": float(best["val_loss"]),
        "final_train_loss": float(final["train_loss"]),
        "final_val_loss": float(final["val_loss"]),
        "final_gap": float(final["generalization_gap"]),
    }

runs = [BASELINE_RUN, IMPROVED_RUN] + [f"branch_lr_{lr}" for lr in LRS]
summary_rows = [row for row in [summarize_run(run) for run in runs] if row is not None]
summary_df = pd.DataFrame(summary_rows).sort_values("best_val_loss")
display(summary_df)
summary_df.to_csv(Path(OUT) / "run_comparison_summary.csv", index=False)


## 12. Best Checkpoint 로드 테스트

마지막 epoch가 아니라 validation loss가 가장 낮았던 best checkpoint를 로드해서 테스트합니다.

In [ ]:
TEST_RUN = IMPROVED_RUN

subprocess.run([
    "python", "scripts/step3_test_gpt.py",
    "--artifact-dir", OUT,
    "--run-name", TEST_RUN,
    "--eval-splits", "val,test",
    "--eval-batches", "100",
    "--prompt", "이 영화",
    "--prompt", "정말",
    "--prompt", "배우의 연기가",
], check=True)


In [ ]:
test_dir = Path(OUT) / "runs" / TEST_RUN / "test"
display(pd.read_csv(test_dir / "split_metrics.csv"))
display(pd.read_csv(test_dir / "generations.csv"))
print((test_dir / "test_report.md").read_text(encoding="utf-8")[:2000])


## 13. Resume Example

Colab 런타임이 끊겼을 때는 같은 run name과 last checkpoint로 이어서 실행합니다. 필요한 경우 아래 셀의 주석을 풀어 사용하세요.

In [ ]:
# RESUME_RUN = IMPROVED_RUN
# LAST_CKPT = str(Path(OUT) / "runs" / RESUME_RUN / "last_checkpoint.pt")
#
# subprocess.run([
#     "python", "scripts/step2_train_gpt.py",
#     "--artifact-dir", OUT,
#     "--run-name", RESUME_RUN,
#     "--num-epochs", "100",
#     "--context-length", "128",
#     "--emb-dim", "192",
#     "--n-heads", "4",
#     "--n-layers", "4",
#     "--batch-size", "16",
#     "--lr", "3e-4",
#     "--min-lr", "1e-5",
#     "--drop-rate", "0.2",
#     "--weight-decay", "0.1",
#     "--scheduler", "cosine",
#     "--eval-freq", "100",
#     "--eval-iter", "20",
#     "--grad-clip", "1.0",
#     "--resume-checkpoint", LAST_CKPT,
# ], check=True)


## 14. 리포트에 넣을 산출물 위치

In [ ]:
print("BPE manifest:", Path(OUT) / "bpe_manifest.json")
print("BPE vocab:", Path(OUT) / "bpe/bpe_vocab_3000.json")
print("token ids:", Path(OUT) / "tokens")
print("run comparison:", Path(OUT) / "run_comparison_summary.csv")
print("epoch table:", Path(OUT) / "runs" / TEST_RUN / "epoch_summary.csv")
print("loss plot:", Path(OUT) / "runs" / TEST_RUN / "loss_curves.png")
print("best checkpoint:", Path(OUT) / "runs" / TEST_RUN / "best_checkpoint.pt")
print("test report:", Path(OUT) / "runs" / TEST_RUN / "test/test_report.md")
